In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import random


In [1]:

class Value:
    def __init__(self, data , _children=() , _op='' , label = ''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data+other.data , (self,other)  , '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 *  out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):  # Handles: 2 + self
        return self + other

    def __mul__(self,other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data*other.data , (self,other) , '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
        return self*-1

    def __sub__(self,other):
        return self + (-other)

    def __rsub__(self,other):
        return other + (-self) # diff than other __r__

    def __rmul__(self, other):  # Handles: 2 * self
        return self * other

    def __truediv__(self,other):
        return self * other**-1

    def __rtruediv__(self,other):
        return other * self**-1

    def __pow__(self, other):
        assert isinstance(
            other, (int, float)
        ), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f"**{other}")
        def _backward():
            self.grad += (other * (self.data ** (other - 1))) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out = Value(t,(self,),'tanh')
        def _backward():
            self.grad += (1-t**2)*out.grad
        out._backward =_backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,) ,'exp')
        def _backward():
            self.grad += out.grad*out.data
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    build_topo(child)
                topo.append(node)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [55]:
from graphviz import Digraph
def trace(root):
    nodes,edges = set(),set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes,edges
def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f}" % (n.label, n.data , n.grad),
        shape='record'
        )
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
a = Value(2.0 , label="a")
b = Value(-3.0 , label="b")
c = Value(10.0 , label="c")
e = a*b;e.label="e"
d = e+c;d.label="d"
f = Value(-2.0,label="f")
L = d*f; L.label="L"

draw_dot(L)

In [87]:
L.grad = 1.0
d.grad = -2.0
f.grad = 4.0
e.grad = -2.0
c.grad = -2.0
a.grad = -2.0 * -3.0
b.grad = -2.0 * 2.0


In [ ]:

plt.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2)))
plt.grid()
plt.show()

In [101]:
# inputs x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')
x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [ ]:

o.grad = 1.0
n.grad = 1-o.data**2
x1w1x2w2.grad = n.grad
b.grad = n.grad
x1w1.grad = n.grad
x2w2.grad = n.grad
x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad
x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad
draw_dot(o)

In [ ]:
o.grad = 1.0
o._backward()
n._backward()
b._backward()
x1w1x2w2._backward()
x1w1._backward()
x2w2._backward()
draw_dot(o)

In [ ]:
o.backward()
draw_dot(o)

In [252]:

class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,-1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,-1))
    def __call__(self,x):
        act = sum((wi*xi for wi,xi in zip(self.w,x)),self.b)
        out = act.tanh()
        return out
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self,x):
        outs = [neuron(x) for neuron in self.neurons]
        return outs[0] if len(outs)==1 else outs
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self,nin,nouts):
        sz = [nin]+nouts
        self.layers = [Layer(sz[i],sz[i+1]) for i in range(len(nouts))]
    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [253]:
x = [2.0 , 3.0 ,-1]
n = MLP(3,[4,4,1])
m = MLP(3,[4,4,1])
# Check if neurons are unique memory addresses
print(n.layers[0].neurons[0])
print(n.layers[0].neurons[1])

# Check if their weight values are identical
print(n.layers[0].neurons[0].w)
print(n.layers[0].neurons[1].w)
n(x),m(x)
len(n.parameters())

[Value(data=-1.0), Value(data=-1.0), Value(data=-1.0)]
[Value(data=-1.0), Value(data=-1.0), Value(data=-1.0)]


41

In [254]:

# 1. Dataset Inputs (4 data points, each with 3 feature values)
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
# Target outputs (desired output for each input vector)
ys = [1.0, -1.0, -1.0, 1.0]


In [249]:
ypred = [n(x) for x in xs]
loss = sum((yout-ygt)**2 for yout , ygt in zip(ypred,ys))
loss


Value(data=7.999239347363645)

In [199]:
loss.backward()

In [200]:
for p in n.parameters():
    p.data-= 0.01 * p.grad


In [283]:
# GRADIENT DESECTN

for k in range(100):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout-ygt)**2 for yout , ygt in zip(ypred,ys))

    # backward pass
    for p in n.parameters():
        p.grad=0.0
    loss.backward()

    #update
    for p in n.parameters():
        p.data -= 0.1 * p.grad

loss.data


0.00010288798364798543

In [284]:
ypred

[Value(data=0.9974808663434869),
 Value(data=-0.9934449946007236),
 Value(data=-0.9931290387201732),
 Value(data=0.997477353656339)]

In [ ]:
draw_dot(loss)